# Investigação de Anomalias

Notebook dedicado ao preenchimento manual de `causa_raiz` para anomalias detectadas
automaticamente pelo `07_auditoria_execucoes` (ver ADR-15). Centraliza esse trabalho,
antes espalhado em células soltas em múltiplos notebooks.

**Fluxo de uso:** sempre que o Dashboard (página "Observabilidade Técnica") mostrar
uma anomalia como "Aguardando investigação", investigar a causa e adicionar um
`UPDATE` aqui, seguindo o padrão de prefixo:

- `[TESTE]` — anomalia causada por desenvolvimento/teste manual do projeto (célula
  por célula, reprocessamento durante debug, etc.) — não reflete o pipeline em
  operação normal.
- `[OPERACIONAL]` — anomalia causada por comportamento real do pipeline em produção
  (API lenta, sincronização tardia de dependência externa, etc.) — merece atenção
  contínua, mesmo com causa já identificada.
- Sem prefixo, começando com `NAO DETERMINADA` — causa não confirmada (padrão já
  usado desde o ADR-15).

Esse prefixo alimenta a métrica de "anomalias de teste vs. operacionais" no Dashboard.

**Este notebook não roda automaticamente via Job** — é de uso manual, sob demanda,
sempre que uma investigação for necessária.

In [0]:
# imports
from pyspark.sql import functions as F

In [0]:
# consulta - anomalias aguardando investigacao (causa_raiz nula)
display(spark.table("poc_b3_modernizacao.observability.auditoria_anomalias")
    .filter(F.col("causa_raiz").isNull())
    .orderBy("inicio")
)

In [0]:
# aplica prefixo [TESTE] retroativamente nas anomalias de desenvolvimento/teste
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = CONCAT('[TESTE] ', causa_raiz)
    WHERE causa_raiz IS NOT NULL
      AND causa_raiz NOT LIKE '[TESTE]%'
      AND causa_raiz NOT LIKE '[OPERACIONAL]%'
      AND causa_raiz NOT LIKE 'NAO DETERMINADA%'
      AND (
        (notebook = '04_gold' AND inicio >= to_timestamp('2026-08-28T14:00:00') AND inicio <= to_timestamp('2026-08-28T15:00:00'))
        OR (inicio >= to_timestamp('2026-09-01T18:18:00') AND inicio <= to_timestamp('2026-09-01T18:40:00'))
        OR (notebook = '07_auditoria_execucoes' AND inicio >= to_timestamp('2026-09-02T18:48:00') AND inicio <= to_timestamp('2026-09-02T18:49:00'))
        OR (notebook = '05_reconciliacao' AND inicio >= to_timestamp('2026-09-02T20:00:00') AND inicio <= to_timestamp('2026-09-02T20:30:00'))
        OR (notebook = '03_silver' AND inicio >= to_timestamp('2026-09-04T14:22:00') AND inicio <= to_timestamp('2026-09-04T14:23:00'))
        OR (notebook = '05_reconciliacao' AND inicio >= to_timestamp('2026-09-04T16:32:00') AND inicio <= to_timestamp('2026-09-04T16:33:00'))
      )
""")

# aplica prefixo [OPERACIONAL] na anomalia real (sincronizacao tardia do KNIME)
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = CONCAT('[OPERACIONAL] ', causa_raiz)
    WHERE causa_raiz IS NOT NULL
      AND causa_raiz NOT LIKE '[TESTE]%'
      AND causa_raiz NOT LIKE '[OPERACIONAL]%'
      AND causa_raiz NOT LIKE 'NAO DETERMINADA%'
      AND inicio >= to_timestamp('2026-08-28T20:28:00') AND inicio <= to_timestamp('2026-08-28T20:32:00')
""")

print("Prefixos aplicados retroativamente.")

In [0]:
display(spark.table("poc_b3_modernizacao.observability.auditoria_anomalias")
    .select("notebook", "inicio", "tipo_anomalia", "causa_raiz")
    .orderBy("inicio")
)

In [0]:
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = CONCAT('NAO DETERMINADA - ', causa_raiz)
    WHERE notebook = '01_ingestao_landing'
      AND inicio >= to_timestamp('2026-09-03T20:15:00') AND inicio <= to_timestamp('2026-09-03T20:16:00')
""")
print("Reclassificado como NAO DETERMINADA.")

In [0]:
display(spark.sql("""
    SELECT
        CASE
            WHEN causa_raiz LIKE 'NAO DETERMINADA%' THEN 'Requer atenção'
            WHEN causa_raiz IS NULL THEN 'Aguardando investigação'
            WHEN causa_raiz LIKE '[TESTE]%' THEN 'Causa conhecida - Teste'
            WHEN causa_raiz LIKE '[OPERACIONAL]%' THEN 'Causa conhecida - Operacional'
            ELSE 'Sem classificacao'
        END AS categoria,
        COUNT(*) AS quantidade
    FROM poc_b3_modernizacao.observability.auditoria_anomalias
    GROUP BY 1
    ORDER BY 1
"""))